In [1]:
import lightgbm as lgb
import pandas as pd
import optuna
import warnings
import json

from i import input_dir, output_dir, model_dir
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error

warnings.filterwarnings("ignore")

train = pd.read_csv(input_dir + "train.csv", index_col="id")
test = pd.read_csv(input_dir + "test.csv", index_col="id")


for col in train.select_dtypes(include="object").columns:
    train[col] = train[col].astype("category")
    test[col] = test[col].astype("category")


Target_Col = "exam_score"

X = train.iloc[:, :-1]
y = train[Target_Col]
X_test = test

c:\Users\Blanc\AppData\Local\pypoetry\Cache\virtualenvs\playground-lwZmZsxv-py3.11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
cat_cols = X.select_dtypes(["category"]).columns.tolist()


def objective(trial):
    param = {
        "objective": "regression",
        "metric": "rmse",
        "verbosity": -1,
        "boosting_type": "gbdt",
        # GPU acceleration
        "device": "gpu",
        "gpu_platform_id": 0,
        "gpu_device_id": 0,
        # Learning
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 8000),
        # Tree structure
        "num_leaves": trial.suggest_int("num_leaves", 16, 512),
        "max_depth": trial.suggest_int("max_depth", -1, 128),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 500),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 10.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 5.0),
        # Regularization
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        # Sampling
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 0, 10),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        # Histogram / tree
        "max_bin": trial.suggest_int("max_bin", 32, 255),
        "grow_policy": trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"]),
        "extra_trees": trial.suggest_categorical("extra_trees", [False, True]),
        "path_smooth": trial.suggest_float("path_smooth", 0.0, 100.0),
        # etra addition
        "min_sum_hessian_in_leaf": trial.suggest_float("min_sum_hessian_in_leaf", 1e-3, 100.0, log=True),
        "force_row_wise": trial.suggest_categorical("force_row_wise", [True, False]),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 0.5, 10.0, log=True),
    }

    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []

    for train_idx, valid_idx in cv.split(X, y):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        model = lgb.LGBMRegressor(**param)

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            eval_metric="rmse",
            categorical_feature=cat_cols,  # Pass categorical columns
            callbacks=[lgb.early_stopping(50), optuna.integration.LightGBMPruningCallback(trial, "rmse")],  # <-- pruning
        )

        preds = model.predict(X_valid)
        aucs.append(root_mean_squared_error(y_valid, preds))

    return sum(aucs) / len(aucs)


# Run the tuner
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(),
    study_name="my_lgbm_study",
    storage=f"sqlite:///{model_dir}optuna_S6E01.db",
    load_if_exists=True,
)
study.optimize(objective, n_trials=200)

print("Best params:", study.best_params)
print("Best AUC:", study.best_value)


[I 2026-01-24 22:23:11,293] Using an existing study with name 'my_lgbm_study' instead of creating a new one.
[I 2026-01-24 22:23:12,946] Trial 201 pruned. Trial was pruned at iteration 0.


Training until validation scores don't improve for 50 rounds


[I 2026-01-24 22:23:16,625] Trial 202 pruned. Trial was pruned at iteration 42.
[I 2026-01-24 22:23:17,187] Trial 203 pruned. Trial was pruned at iteration 0.


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[145]	valid_0's rmse: 8.74744
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[157]	valid_0's rmse: 8.75367
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[149]	valid_0's rmse: 8.7495
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[143]	valid_0's rmse: 8.76763
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[180]	valid_0's rmse: 8.78368


[I 2026-01-24 22:24:29,244] Trial 204 finished with value: 8.760427875864703 and parameters: {'learning_rate': 0.13853054054187372, 'n_estimators': 4328, 'num_leaves': 302, 'max_depth': 43, 'min_data_in_leaf': 492, 'min_child_weight': 0.12776977737640832, 'min_split_gain': 4.637392481590193, 'lambda_l1': 0.00015437343745216476, 'lambda_l2': 0.0015887637366580635, 'bagging_fraction': 0.9905161762641559, 'bagging_freq': 3, 'feature_fraction': 0.6406785214821098, 'max_bin': 240, 'grow_policy': 'depthwise', 'extra_trees': False, 'path_smooth': 65.16949629911647, 'min_sum_hessian_in_leaf': 0.01319123481598139, 'force_row_wise': False, 'scale_pos_weight': 2.0747967808375867}. Best is trial 4 with value: 8.74900437204158.


...

Training until validation scores don't improve for 50 rounds


[I 2026-01-24 22:59:50,644] Trial 398 pruned. Trial was pruned at iteration 50.
[I 2026-01-24 22:59:51,394] Trial 399 pruned. Trial was pruned at iteration 0.


Training until validation scores don't improve for 50 rounds


[I 2026-01-24 22:59:54,479] Trial 400 pruned. Trial was pruned at iteration 32.


Best params: {'learning_rate': 0.0277993931187874, 'n_estimators': 5014, 'num_leaves': 344, 'max_depth': 59, 'min_data_in_leaf': 89, 'min_child_weight': 0.8945335588186097, 'min_split_gain': 3.8029204324335515, 'lambda_l1': 1.3864864756259642e-05, 'lambda_l2': 0.5825549352737858, 'bagging_fraction': 0.8186727976415629, 'bagging_freq': 7, 'feature_fraction': 0.5116646618518729, 'max_bin': 249, 'grow_policy': 'lossguide', 'extra_trees': False}
Best AUC: 8.74900437204158


In [8]:
best_params = study.best_params
with open(model_dir + "lgbm_base_params.json", "w") as f:
    json.dump(best_params, f, indent=4)